<a href="https://colab.research.google.com/github/codeBMB/google_colabs/blob/main/python_colabs/PDB_Advanced_Search_Drugs_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="150" height="150" />

# Project: Advanced Search for Approved Drugs in the RCSB PDB
## Notebook: PDB_Advanced_Search_Drugs_Colab.ipynb

**Purpose:**
We will use some python scripting and python libraries to query the RCSB PDB for the structures of approved drugs that are found in RCSB PDB protein structure files. We will then look at the code needed to visualize the protein-drug interactions.

**Input Data:**
* **Description** The data for this notebook will be extracted using the RCSB PDB API. There are no data files to download directly
* **Source:** [RCSB PDB](https://www.rcsb.org)
* **Retrieved On:** N/A
* **Access:** N/A

**Libraries**
| Library | Abbreviation | Citation |
|----------|----------|----------|
| rcsb-api | rcsb-api | https://pypi.org/project/rcsb-api/ |
| requests | requests | https://pypi.org/project/requests/ |
| os | os | https://docs.python.org/3/library/os.html |

**Learning Objectives**

The user will recognized and execute code to
- search the [RCSB PDB](https://www.rcsb.org) for approved drugs that are bound to structures there;
- extract the chemical and trade names of the approved drugs
- identify protein structures that contain the drugs
- visualize a protein structure with the bound drug

## **Note!**
You should make your own copy of this Colab notebook (navigate to **File > Save a Copy in Drive** and write your code in that notebook as you follow along with the lesson. Make sure to run every single code cell you come across.

**Status with Date:** [Work in Progress]
For internal development purposes. Remove this section in the shared notebooks for users.

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100" alttext="[CC BY-NC-SA](https://creativecommons.org/licenses/by-nc-sa/4.0/)"/>
 This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build upon the material, you must license the modified material under identical terms. CC BY-NC-SA includes the following elements:
 BY: credit must be given to the creator.
 NC: Only noncommercial uses of the work are permitted.
 SA: Adaptations must be shared under the same terms.

---
**Authorship:** Paul A. Craig | https://github.com/paulcraig

**Acknowledgements:** [NSF 218732, 218733](https://www.nsf.gov/awardsearch/show-award/?AWD_ID=2518732) <a href="https://www.rcsb.org"><img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/rgb-logo.png" width="80" alttext="[PDB Logo]()"/>
<a href="https://molssi.org/"><img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/molssi_main_logo-300x204.png" width="50" alttext="[PDB Logo]()"/>

**Contact Info:** paul.craig@rit.edu

# 0. Explanation of Colab and how to run (in notebooks for the first workshop)

To run the cell below, simply click the Run button located in that cell.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

NOTE: A cell is still running if you see a "stop" button with a moving circle around it. You can tell a cell has completed running as it will have a number in brackets on the left hand side (e.g. [1]) and a checkmark with the amount of time it took to run underneath it.

Please ensure every cell is done running before running the next cell.

# 1. Setup

In the Colab environment, cells are executed in the order you select. Normally this is done from top to bottom, but you can use any order you like. However, you must have acces to the correct libraries, so it is best practice to do that first for all of the libraries you need, rather than installing and importing them as they are needed. In the next two cells you will install the libraries you need for this notebook and import them into your Colab environment.

In [ ]:
%%capture
!pip install requests rcsb-api py3Dmol

In [ ]:
from rcsbapi.search import AttributeQuery, TextQuery
from rcsbapi.search import search_attributes as attrs
import requests  # to enable us to pull files from the PDB
import os        # to enable us to create a directory to store the files
import py3Dmol   # for visualizing structures

# 2. Searching to Drugs Approved for the US Market

It is natural to search the RCSB PDB for protein structures, since protein is in the title. However, some of the most interesting information there is about the small molecules that are bound to the macromolecular structures - substrates, inhibitors, metal ions and drug molecules.

In the next cell, we'll perform a query of the RCSB PDB to find approved drugs that are bound to structures there. This two-step query begins with a search for drugs that have been approved, followed by a query by country. These two queries are combined in a single statement and then the results are specified for `mol_definition`, which gives the PDB ID for the drug molecules themselves.

Once you have executed this cell look for drugs that have been approved for Canada and the EU. Are there more approved drugs for the country you selected than for the US market? Try full country names or abbreviations.

In [ ]:
market_approved = "Y" #this variable indicates certified approval at a national level
country = "US" #this variable indicates which country is being referred to

q1 = AttributeQuery(attribute="drugbank_info.drug_products.approved", operator="exact_match", value=market_approved)
#Attribute for market approval
q2 = AttributeQuery(attribute="drugbank_info.drug_products.country", operator="exact_match", value=country)
#Attribute for country
query = q1 & q2 #Combining attributes into a single query
result_drugs = list(query("mol_definition"))
print(result_drugs)

In [ ]:
print(f"The following drugs are among the {len(result_drugs)} drugs approved for the market in the {country}: {result_drugs[0:10]}")

# 3. Finding the Names of the Drugs

Now that we know how many drugs approved for the US market are bound to structures in the RCSB PDB, let's find the names of a few drugs. Explore the code below for loops and logic, then find the key terms that are used to extract the drug names.

In [ ]:
drug_names = {}
base_url = "https://files.rcsb.org/ligands/download/"

# Let's get the names for the first 5 drugs as an example
for drug_id in result_drugs[0:10]:
    cif_url = base_url + f"{drug_id}.cif"
    response = requests.get(cif_url)

    if response.status_code == 200:
        cif_content = response.text
        # Parse the CIF content to find the drug name
        for line in cif_content.splitlines():
            if line.strip().startswith('_chem_comp.name'):
                # Extract the name, removing the keyword and any quotes/whitespace
                name = line.split(' ', 1)[1].strip().strip('"')
                drug_names[drug_id] = name
                break # Found the name, move to the next drug_id
    else:
        drug_names[drug_id] = "Name not found (Error retrieving CIF)"

print("Drug names for selected mol_definitions:")
for drug_id, name in drug_names.items():
    print(f"ID: {drug_id}, Name: {name}")

The names we found were highly technical and are not commonly recognized. Create a new cell using the text from the previous cell. You'll need to replace a term in that code with the term `pdbx_synonyms` to find some familiar names.

In [ ]:
# Paste the code from the previous cell here and modify it to look for common drug names.



In [ ]:
# @title
drug_trade_names = {}
base_url = "https://files.rcsb.org/ligands/download/"

# Let's get the trade names for the first 5 drugs as an example
for drug_id in result_drugs[0:10]:
    cif_url = base_url + f"{drug_id}.cif"
    response = requests.get(cif_url)

    if response.status_code == 200:
        cif_content = response.text
        # Parse the CIF content to find the trade name
        trade_name_found = False
        for line in cif_content.splitlines():
            if line.strip().startswith('_chem_comp.pdbx_synonyms'):
                # Extract the name, removing the keyword and any quotes/whitespace
                name = line.split(' ', 1)[1].strip().strip('"')
                drug_trade_names[drug_id] = name
                trade_name_found = True
                break # Found the name, move to the next drug_id
        if not trade_name_found:
            drug_trade_names[drug_id] = "Trade name not found in CIF"
    else:
        drug_trade_names[drug_id] = "Trade name not found (Error retrieving CIF)"

print("Trade names for selected mol_definitions:")
for drug_id, name in drug_trade_names.items():
    print(f"ID: {drug_id}, Trade Name: {name}")

# 4. Downloading the Drug Structure Files
Before visualizing the drug structures from this list, let's try to download one and see how it goes. In the following cells, we'll run a test to see if the download was successful. Now would be a good time to open the Files menu on the left side of the page to see when the new `ligands` folder appears.

In [ ]:
#Downloading a file from our list:

test_validation = requests.get(f'https://files.rcsb.org/ligands/download/{result_drugs[0]}.cif')

In [ ]:
# check to see that the file downloaded properly. A status code of 200 means everything is okay.

test_validation.status_code        # Status code check

In [ ]:
# To really be sure, let's look at the file one line at a time. First we write the downloaded content to a file.

# make a ligands folder for our results. If this ligands folder already exists, then it doesn't create a new one
os.makedirs("ligands", exist_ok=True)

with open(f"ligands/{result_drugs[0]}.cif", "w+") as file:
    file.write(test_validation.text)

In [ ]:
file1 = open(f'ligands/{result_drugs[0]}.cif', 'r')
file_text = file1.read() # This reads in the file as a string.

print(file_text)

**Potential Time Sink**

If you execute the next cell, it will take 5-10 minutes to download all the drug structures. Please hold off on that for a moment.

In [ ]:
baseUrl = "https://files.rcsb.org/ligands/download/"

os.makedirs("ligands/USMarketDrugSearch/", exist_ok=True)

for ChemID in result_drugs:
    cFile = f"{ChemID}.cif"
    cFileUrl = baseUrl + cFile
    cFileLocal = "ligands/USMarketDrugSearch/" + cFile
    response = requests.get(cFileUrl)
    with open(cFileLocal, "w+") as file:
        file.write(response.text)

# 5. Finding proteins bound to Olaparib

Now, let's find the PDB entries that contain Midazolam, which has the `RCSB PDB ID 09L`. This code was generated with assistance from Gemini, using the `TextQuery` function from the `rcsb-api` library.

In [ ]:
olaparib_id = '09L'

# Use TextQuery to find entries containing the ligand ID
# TextQuery implicitly uses the 'text' service and searches across relevant fields.
query_text = TextQuery(olaparib_id)

# Perform the query and explicitly request 'entry' as the return type
proteins_bound_to_olaparib = list(query_text("entry"))

print(f"PDB entries containing Olaparib (ID: {olaparib_id}):")
if proteins_bound_to_olaparib:
    print(proteins_bound_to_olaparib)
else:
    print("No PDB entries found for Olaparib.")


In [ ]:
# @title
# This code enables you to find PDB entries with only one chain

import requests # Ensure requests is imported if not already globally available

single_chain_olaparib_proteins = []

for pdb_id in proteins_bound_to_olaparib:
    # Construct the URL to fetch detailed entry data for each PDB ID
    entry_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    response = requests.get(entry_url)

    if response.status_code == 200:
        entry_data = response.json()
        # Access the polymer_entity_count_macro from the entry data
        polymer_entity_count = entry_data.get('rcsb_entry_info', {}).get('polymer_entity_count_macro')

        # If the count is 1, add it to our list
        if polymer_entity_count == 1:
            single_chain_olaparib_proteins.append(pdb_id)
    else:
        print(f"Warning: Could not retrieve data for PDB ID {pdb_id}. Status code: {response.status_code}")

print(f"Proteins bound to Olaparib with only one chain: {single_chain_olaparib_proteins}")

# 6. Visualizing a Drug Molecule Bound to a Target Protein

You are probably familiar with visualization software tools such as PyMOL, Chimera, JSMol or Mol*. The `py3Dmol` library is a Python adaptation of JSmol. We installed and imported `py3Dmol` very early in this notebook.

Now we will use it to visualize a protein that contains the drug molecule, [olaparib](https://pubchem.ncbi.nlm.nih.gov/compound/23725625), which is used to treat ovarian cancer.

1. Find the proteins that are bound to the Olaparib.
1. Visualize the protein-drug complex with py3Dmol.
1. Make some tweaks to the structure to make it more interesting.

The visualization below is adapted from a notebook that is an abbreviated and adapted version of [Lab 02 of IIBM3202 Molecular Modeling and Simulation](https://github.com/pb3lab/ibm3202/blob/master/tutorials/lab02_molviz.ipynb) from the Institute for Biological and Engineering at Pontificia Universidad Catolica de Chile.

**Visualizing molecular structures with py3DMol**

In this computer lab, we will visualize the structures of various biomolecules using [py3DMol](https://github.com/avirshup/py3dmol). py3DMol which allows for dependency-free molecular visualization in Jupyter notebooks. py3DMol wraps the [3DMol.js](http://3dmol.csb.pitt.edu/doc/index.html) library for online molecular visualization.

This lab is an abbreviated and adapted version of [Lab 02 of IIBM3202 Molecular Modeling and Simulation](https://github.com/pb3lab/ibm3202/blob/master/tutorials/lab02_molviz.ipynb) from the Institute for Biological and Engineering at Pontificia Universidad Catolica de Chile.

**Retrieving structures**

In a cell above, we have a list of structures that contain Olaparib from the [Protein Data Bank](https://www.rcsb.org/), including [4TKG](https://www.rcsb.org/structure/4TKG).

You can manually retrieve PDB structures from its website (www.rcsb.org). Alternatively, you can directly use the terminal to download a given PDB file with known accession code as shown below, where XXXX must be replaced by the replaced by the 4-letter PDB code:

```
!wget http://www.rcsb.org/pdb/files/XXXX.pdb.gz
!gunzip XXXX.pdb.gz
```

In [ ]:
!wget http://www.rcsb.org/pdb/files/8hlj.pdb.gz
!gunzip 8hlj.pdb.gz

**Styles**

Now let's visualize the structure. In the code below, notice that we first add a <i>model</i> and then add <i>styles</i>. The `view.addStyle` function requires an [atom selection](https://3dmol.csb.pitt.edu/doc/types.html#AtomSpec) and a [style specification](https://3dmol.csb.pitt.edu/doc/types.html#AtomStyleSpec). The types of styles include `line`, `stick`, `sphere`, and `cartoon` and properties of the styles include [colorscheme](https://3dmol.csb.pitt.edu/doc/types.html#ColorschemeSpec) or specific [color](https://3dmol.csb.pitt.edu/doc/types.html#ColorSpec).

In [ ]:
# First pass with code from another notebook

view = py3Dmol.view()
view.addModel(open('8hlj.pdb', 'r').read(),'pdb')
view.setBackgroundColor('white')
view.setStyle({'chain':'A'}, {'cartoon': {'color':'purple'}})
view.addStyle({'resn':'09L'}, {'stick': {'colorscheme':'yellowCarbon'}})
view.addStyle({'within':{'distance':'5', 'sel':{'resn':'09L'}}}, {'stick': {}})
view.zoomTo()
view.show()

**Focusing on one chain**

The image above has one chain we want to explore, so let's try to remove the other parts of the structure.

In [ ]:
# Set the starting conditions
view = py3Dmol.view()
view.addModel(open('8hlj.pdb', 'r').read(),'pdb')
view.setBackgroundColor('white')

# Hide everything by default, to ensure only selected parts are visible
view.setStyle({}, {'hide':{}})

# Display chain A as a purple cartoon
view.setStyle({'chain':'A'}, {'cartoon': {'color':'purple'}})

# Display ligand 09L only in chain A as yellow carbon sticks
view.addStyle({'resn':'09L', 'chain':'A'}, {'stick': {'colorscheme':'yellowCarbon'}})

# Display residues in chain A within 5 Angstroms of ligand 09L (in chain A) as sticks
view.addStyle({'chain':'A', 'within':{'distance':5, 'sel':{'resn':'09L', 'chain':'A'}}}, {'stick': {}})

# Bring the selected model to the center of the view screen
view.zoomTo()